# Module 2: Advanced Linear Algebra

Now we go deeper! This module covers the linear algebra that makes modern AI possible — from PCA for dimensionality reduction to SVD for matrix factorization.

### 🎯 What you'll learn:
- Eigenvalues and eigenvectors
- Eigendecomposition and diagonalization
- Singular Value Decomposition (SVD)
- Principal Component Analysis (PCA)
- Matrix factorizations (LU, QR, Cholesky)
- Positive definite matrices
- Orthogonality and Gram-Schmidt process

### 🤖 Why it matters for AI:
- **PCA** is the most used dimensionality reduction technique
- **SVD** powers recommendation systems (Netflix Prize!)
- **Eigenvalues** determine training stability of neural networks
- **Cholesky** decomposition used in Gaussian processes
- **LoRA** (Low-Rank Adaptation) uses SVD concepts to fine-tune LLMs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-darkgrid')

---
## 1. Eigenvalues and Eigenvectors

An **eigenvector** of a matrix $A$ is a non-zero vector $\mathbf{v}$ that, when multiplied by $A$, only gets scaled (not rotated):

$$A\mathbf{v} = \lambda \mathbf{v}$$

- $\mathbf{v}$ is the **eigenvector** (the direction that doesn't change)
- $\lambda$ is the **eigenvalue** (the scaling factor)

### Finding Eigenvalues
Solve: $\det(A - \lambda I) = 0$ (the **characteristic equation**)

### 🤖 AI Connection:
- Eigenvalues of the Hessian matrix tell you about the **curvature** of the loss landscape
- Large eigenvalue ratio → the loss surface is a narrow valley (hard to optimize)
- **Google's PageRank** is an eigenvector problem!

In [ ]:
A = np.array([[4, 1],
              [2, 3]])

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Matrix A:\n{A}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

# Verify: A @ v = λ * v for each eigenpair
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    lhs = A @ v
    rhs = lam * v
    print(f"\nEigenpair {i+1}: λ = {lam:.4f}")
    print(f"  A @ v = {np.round(lhs, 4)}")
    print(f"  λ * v = {np.round(rhs, 4)}")
    print(f"  Equal? {np.allclose(lhs, rhs)}")

# Visualize eigenvectors
fig, ax = plt.subplots(figsize=(8, 8))

# Draw the effect of A on many vectors
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])
ellipse = A @ circle

ax.plot(circle[0], circle[1], 'b-', alpha=0.3, label='Unit circle')
ax.plot(ellipse[0], ellipse[1], 'r-', alpha=0.3, label='A × unit circle')

# Draw eigenvectors
for i in range(2):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, 
             color=f'C{i+2}', linewidth=3, label=f'v{i+1} (λ={lam:.2f})')
    ax.quiver(0, 0, lam*v[0], lam*v[1], angles='xy', scale_units='xy', scale=1,
             color=f'C{i+2}', linewidth=1, alpha=0.5, linestyle='dashed')

ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.legend(fontsize=11); ax.set_title('Eigenvectors: Directions Preserved by A', fontsize=14)
plt.show()

---
## 2. Eigendecomposition (Diagonalization)

If $A$ has $n$ linearly independent eigenvectors, we can write:

$$A = Q \Lambda Q^{-1}$$

Where:
- $Q$ = matrix of eigenvectors (as columns)
- $\Lambda$ = diagonal matrix of eigenvalues

### Why is this useful?
- **Matrix powers**: $A^k = Q \Lambda^k Q^{-1}$ (just raise eigenvalues to power $k$!)
- **Matrix exponential**: $e^A = Q e^\Lambda Q^{-1}$

### For symmetric matrices ($A = A^T$):
$$A = Q \Lambda Q^T$$
The eigenvectors are **orthogonal**, so $Q^{-1} = Q^T$ (much simpler!).

In [ ]:
# Eigendecomposition
A = np.array([[4, 1],
              [2, 3]])

eigenvalues, Q = np.linalg.eig(A)
Lambda = np.diag(eigenvalues)

# Reconstruct A from eigendecomposition
A_reconstructed = Q @ Lambda @ np.linalg.inv(Q)

print(f"A =\n{A}")
print(f"\nQ (eigenvectors) =\n{np.round(Q, 4)}")
print(f"\nΛ (eigenvalues) =\n{np.round(Lambda, 4)}")
print(f"\nQ Λ Q⁻¹ =\n{np.round(A_reconstructed, 4)}")
print(f"\nReconstruction matches? {np.allclose(A, A_reconstructed)}")

# Symmetric matrix — eigenvectors are orthogonal!
S = np.array([[2, 1],
              [1, 3]])
eigenvalues_s, Q_s = np.linalg.eigh(S)  # Use eigh for symmetric!
print(f"\nSymmetric matrix eigenvalues: {np.round(eigenvalues_s, 4)}")
print(f"Q^T @ Q = I? {np.allclose(Q_s.T @ Q_s, np.eye(2))}")

---
## 3. Singular Value Decomposition (SVD)

SVD is arguably the **most important matrix factorization** in all of machine learning.

**Any** matrix $A \in \mathbb{R}^{m \times n}$ can be decomposed as:

$$A = U \Sigma V^T$$

Where:
- $U \in \mathbb{R}^{m \times m}$ — left singular vectors (orthogonal)
- $\Sigma \in \mathbb{R}^{m \times n}$ — diagonal matrix of **singular values** ($\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$)
- $V^T \in \mathbb{R}^{n \times n}$ — right singular vectors (orthogonal)

### Low-Rank Approximation
Keep only the top $k$ singular values to approximate $A$:
$$A \approx U_k \Sigma_k V_k^T$$

This is the **best rank-$k$ approximation** (Eckart-Young theorem).

### 🤖 AI Connection:
- **LoRA** fine-tunes LLMs by learning low-rank updates: $W' = W + BA$ where $B, A$ are low-rank
- **Image compression** using truncated SVD
- **Latent Semantic Analysis** in NLP

In [ ]:
# SVD of a matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10, 11, 12]])

U, sigma, Vt = np.linalg.svd(A)

print(f"A ({A.shape}):")
print(A)
print(f"\nU ({U.shape}):")
print(np.round(U, 4))
print(f"\nSingular values: {np.round(sigma, 4)}")
print(f"\nV^T ({Vt.shape}):")
print(np.round(Vt, 4))

# Reconstruct A
Sigma = np.zeros_like(A, dtype=float)
np.fill_diagonal(Sigma, sigma)
A_reconstructed = U @ Sigma @ Vt
print(f"\nReconstruction matches? {np.allclose(A, A_reconstructed)}")

# Low-rank approximation (keep top k=1 singular value)
k = 1
A_approx = U[:, :k] @ np.diag(sigma[:k]) @ Vt[:k, :]
print(f"\nRank-1 approximation:\n{np.round(A_approx, 2)}")
print(f"Approximation error (Frobenius): {np.linalg.norm(A - A_approx):.4f}")

---
## 4. Principal Component Analysis (PCA)

PCA finds the directions of **maximum variance** in data. It's the most common dimensionality reduction technique.

### Algorithm:
1. Center the data: $X_{centered} = X - \bar{X}$
2. Compute covariance matrix: $C = \frac{1}{n-1} X_{centered}^T X_{centered}$
3. Compute eigenvectors of $C$ (or use SVD)
4. Project onto top-$k$ eigenvectors

### 🤖 AI Connection:
- Reduce 1000-dimensional data to 50 dimensions before training
- Visualize high-dimensional embeddings in 2D
- Feature extraction and noise reduction

In [ ]:
# Generate correlated 2D data
np.random.seed(42)
mean = [0, 0]
cov = [[3, 2], [2, 2]]  # Correlated!
X = np.random.multivariate_normal(mean, cov, 200)

# Step 1: Center the data
X_centered = X - X.mean(axis=0)

# Step 2: Covariance matrix
C = np.cov(X_centered.T)
print(f"Covariance matrix:\n{np.round(C, 4)}")

# Step 3: Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(C)

# Sort by decreasing eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f"\nEigenvalues: {np.round(eigenvalues, 4)}")
print(f"Variance explained: {np.round(eigenvalues / eigenvalues.sum() * 100, 2)}%")

# Step 4: Project onto first principal component
X_pca = X_centered @ eigenvectors[:, :1]  # Project to 1D

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original data with principal components
axes[0].scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.3, s=20, color='#3498DB')
for i in range(2):
    v = eigenvectors[:, i] * eigenvalues[i] * 0.5
    axes[0].quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
                  color=['#E74C3C', '#2ECC71'][i], linewidth=3, label=f'PC{i+1} (var={eigenvalues[i]:.2f})')
axes[0].set_xlabel('x₁'); axes[0].set_ylabel('x₂')
axes[0].set_title('Data with Principal Components', fontsize=14)
axes[0].legend(); axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

# Projected data
axes[1].scatter(X_pca[:, 0], np.zeros_like(X_pca[:, 0]), alpha=0.3, s=20, color='#E74C3C')
axes[1].set_xlabel('PC1'); axes[1].set_title('Projected onto PC1 (1D)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Matrix Factorizations: LU, QR, Cholesky

### LU Decomposition
$$A = LU$$
- $L$ = lower triangular, $U$ = upper triangular
- Used for solving systems of equations efficiently

### QR Decomposition
$$A = QR$$
- $Q$ = orthogonal matrix, $R$ = upper triangular
- Used in least squares problems and eigenvalue algorithms

### Cholesky Decomposition
For symmetric positive definite $A$:
$$A = LL^T$$
- $L$ = lower triangular
- **Twice as fast** as LU

### 🤖 AI Connection:
- **Cholesky** is essential for sampling from multivariate Gaussians (used in VAEs, GPs)
- **QR** used in stable training algorithms

In [ ]:
from scipy import linalg

A = np.array([[2, 1, 1],
              [4, 3, 3],
              [8, 7, 9]])

# LU Decomposition
P, L, U = linalg.lu(A)
print("=== LU Decomposition ===")
print(f"L (lower triangular):\n{np.round(L, 4)}")
print(f"U (upper triangular):\n{np.round(U, 4)}")
print(f"P @ L @ U == A? {np.allclose(P @ L @ U, A)}")

# QR Decomposition
Q, R = np.linalg.qr(A)
print(f"\n=== QR Decomposition ===")
print(f"Q (orthogonal):\n{np.round(Q, 4)}")
print(f"R (upper triangular):\n{np.round(R, 4)}")
print(f"Q @ R == A? {np.allclose(Q @ R, A)}")
print(f"Q^T @ Q = I? {np.allclose(Q.T @ Q, np.eye(3))}")

# Cholesky Decomposition (needs positive definite matrix!)
S = np.array([[4, 2, 1],
              [2, 5, 3],
              [1, 3, 6]])  # Symmetric positive definite
L_chol = np.linalg.cholesky(S)
print(f"\n=== Cholesky Decomposition ===")
print(f"L:\n{np.round(L_chol, 4)}")
print(f"L @ L^T == S? {np.allclose(L_chol @ L_chol.T, S)}")

---
## 6. Positive Definite Matrices

A symmetric matrix $A$ is **positive definite** if:
$$\mathbf{x}^T A \mathbf{x} > 0 \quad \text{for all } \mathbf{x} \neq \mathbf{0}$$

Equivalent conditions:
- All eigenvalues are positive
- All pivots are positive
- Cholesky decomposition exists

**Positive semi-definite**: $\mathbf{x}^T A \mathbf{x} \geq 0$ (eigenvalues $\geq 0$)

### 🤖 AI Connection:
- **Covariance matrices** are always positive semi-definite
- The **Hessian** being positive definite means you're at a local minimum
- **Kernel matrices** in SVMs must be positive semi-definite

In [ ]:
# Positive definite matrix
A_pd = np.array([[2, 1],
                 [1, 3]])

eigenvalues = np.linalg.eigvalsh(A_pd)
print(f"Matrix:\n{A_pd}")
print(f"Eigenvalues: {eigenvalues}")
print(f"All positive? {np.all(eigenvalues > 0)} → Positive Definite!")

# Test x^T A x > 0 for random vectors
for _ in range(5):
    x = np.random.randn(2)
    quad_form = x @ A_pd @ x
    print(f"  x = {np.round(x, 3)}, x^T A x = {quad_form:.4f} > 0? {quad_form > 0}")

# Indefinite matrix (has positive AND negative eigenvalues)
A_indef = np.array([[1, 3],
                    [3, 1]])
eigenvalues_indef = np.linalg.eigvalsh(A_indef)
print(f"\nIndefinite matrix eigenvalues: {eigenvalues_indef}")
print(f"Has both positive and negative → Indefinite (saddle point!)")

---
## 7. Orthogonality and Gram-Schmidt Process

### Orthogonal Vectors
$\mathbf{u}$ and $\mathbf{v}$ are orthogonal if $\mathbf{u} \cdot \mathbf{v} = 0$.

### Orthonormal Vectors
Orthogonal AND unit length: $\mathbf{u} \cdot \mathbf{v} = 0$ and $\|\mathbf{u}\| = \|\mathbf{v}\| = 1$.

### Gram-Schmidt Process
Transform any set of linearly independent vectors into an orthonormal set:

Given $\{\mathbf{v}_1, \mathbf{v}_2, \ldots\}$, produce orthonormal $\{\mathbf{e}_1, \mathbf{e}_2, \ldots\}$:

$$\mathbf{u}_1 = \mathbf{v}_1, \quad \mathbf{e}_1 = \frac{\mathbf{u}_1}{\|\mathbf{u}_1\|}$$

$$\mathbf{u}_2 = \mathbf{v}_2 - (\mathbf{v}_2 \cdot \mathbf{e}_1)\mathbf{e}_1, \quad \mathbf{e}_2 = \frac{\mathbf{u}_2}{\|\mathbf{u}_2\|}$$

$$\mathbf{u}_k = \mathbf{v}_k - \sum_{j=1}^{k-1}(\mathbf{v}_k \cdot \mathbf{e}_j)\mathbf{e}_j$$

In [ ]:
def gram_schmidt(V):
    """Gram-Schmidt orthogonalization.
    V: matrix where each column is a vector.
    Returns: Q, orthonormal matrix.
    """
    n = V.shape[1]
    Q = np.zeros_like(V, dtype=float)
    
    for i in range(n):
        q = V[:, i].astype(float)
        for j in range(i):
            q -= np.dot(Q[:, j], V[:, i]) * Q[:, j]
        Q[:, i] = q / np.linalg.norm(q)
    
    return Q

# Test with 3 vectors
V = np.array([[1, 1, 0],
              [1, 0, 1],
              [0, 1, 1]])

Q = gram_schmidt(V)
print(f"Original vectors:\n{V}")
print(f"\nOrthonormal vectors:\n{np.round(Q, 4)}")
print(f"\nQ^T @ Q (should be I):\n{np.round(Q.T @ Q, 10)}")

# Verify orthogonality
for i in range(3):
    for j in range(i+1, 3):
        print(f"q{i+1} · q{j+1} = {np.dot(Q[:, i], Q[:, j]):.10f}")

---
## 8. SVD for Image Compression (Practical Example)

Let's use SVD to compress an image by keeping only the top-$k$ singular values. This demonstrates how a **low-rank approximation** works in practice — the same principle behind LoRA for LLM fine-tuning!

In [ ]:
# Create a sample grayscale image (gradient with patterns)
np.random.seed(42)
size = 100
x = np.linspace(0, 4*np.pi, size)
y = np.linspace(0, 4*np.pi, size)
X, Y = np.meshgrid(x, y)
image = np.sin(X) * np.cos(Y) + 0.5 * np.sin(2*X + Y)

# SVD
U, sigma, Vt = np.linalg.svd(image)

# Reconstruct with different ranks
ranks = [1, 5, 10, 20, 50]

fig, axes = plt.subplots(1, len(ranks) + 1, figsize=(20, 4))

axes[0].imshow(image, cmap='viridis')
axes[0].set_title(f'Original\n(rank {np.linalg.matrix_rank(image)})', fontsize=11)
axes[0].axis('off')

for i, k in enumerate(ranks):
    approx = U[:, :k] @ np.diag(sigma[:k]) @ Vt[:k, :]
    error = np.linalg.norm(image - approx) / np.linalg.norm(image) * 100
    compression = (k * (size + size + 1)) / (size * size) * 100
    axes[i+1].imshow(approx, cmap='viridis')
    axes[i+1].set_title(f'Rank {k}\n({compression:.1f}% storage, {error:.1f}% error)', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('SVD Image Compression — Same Principle as LoRA!', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

# Singular value spectrum
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(sigma, 'o-', color='#E74C3C', markersize=4)
ax.set_xlabel('Index', fontsize=14)
ax.set_ylabel('Singular Value (log scale)', fontsize=14)
ax.set_title('Singular Value Spectrum — Fast Decay = Good Compression', fontsize=14)
ax.grid(True, alpha=0.3)
plt.show()

---
## 9. Norms Revisited — Matrix Norms

### Frobenius Norm
$$\|A\|_F = \sqrt{\sum_{i,j} A_{ij}^2} = \sqrt{\text{tr}(A^T A)} = \sqrt{\sum \sigma_i^2}$$

### Spectral Norm (Operator Norm)
$$\|A\|_2 = \sigma_{\max}(A)$$
The largest singular value. This is the maximum amount $A$ can stretch a vector.

### Nuclear Norm (Trace Norm)
$$\|A\|_* = \sum \sigma_i$$
Sum of singular values. Used in **matrix completion** problems.

### 🤖 AI Connection:
- **Spectral normalization** constrains the spectral norm of weight matrices in GANs
- **Nuclear norm** regularization encourages low-rank solutions

In [ ]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

_, sigma, _ = np.linalg.svd(A)

# Frobenius norm
frob_direct = np.linalg.norm(A, 'fro')
frob_svd = np.sqrt(np.sum(sigma**2))
print(f"Frobenius norm: {frob_direct:.4f} (from SVD: {frob_svd:.4f})")

# Spectral norm (largest singular value)
spectral = np.linalg.norm(A, 2)
print(f"Spectral norm: {spectral:.4f} (σ_max = {sigma[0]:.4f})")

# Nuclear norm (sum of singular values)
nuclear = np.linalg.norm(A, 'nuc')
nuclear_svd = np.sum(sigma)
print(f"Nuclear norm: {nuclear:.4f} (from SVD: {nuclear_svd:.4f})")

---
## 10. Summary: The Linear Algebra Toolbox for AI

| Concept | Where in AI |
|---------|------------|
| Eigenvalues | Loss landscape curvature, PageRank, spectral clustering |
| SVD | LoRA fine-tuning, recommendations, compression |
| PCA | Dimensionality reduction, visualization |
| Cholesky | Gaussian processes, sampling multivariate normals |
| Positive definite | Convexity, kernel functions, covariance |
| Orthogonality | Gram-Schmidt in attention, orthogonal regularization |
| Matrix norms | Spectral normalization (GANs), regularization |

**You now have the full linear algebra toolkit for AI. Next up: Calculus!** 🚀